SCRAPING CODE  - THIS TAKES 9 HOURS TO RUN

In [ ]:
import pandas as pd
import requests
import zipfile
import io
import os
from datetime import datetime, timedelta

# ========== Config ==========

gkg_folder = r"C:\Users\dsp7\Box\MY RESEARCH\India Sentiment Project\GKG_Data"
os.makedirs(gkg_folder, exist_ok=True)

# Date range (for test, you can set start_date = end_date for one day)
start_date = datetime(2024, 1, 1)
end_date = datetime(2024, 11, 30)

flood_keywords = [
    'FLOOD', 'FLOODS', 'FLOODING', 'FLASH FLOOD', 'FLASH FLOODS', 'FLOODING EVENT',
    'FLOOD-RELATED', 'FLOOD DAMAGE', 'RIVER FLOOD', 'RIVER FLOODS', 'URBAN FLOODING',
    'MONSOON FLOOD', 'MONSOON FLOODS', 'HEAVY RAINFALL', 'HEAVY RAIN', 'INUNDATION',
    'WATER LOGGING', 'WATERLOGGING', 'FLOODING-RELATED'
]
drought_keywords = [
    'DROUGHT', 'DROUGHTS', 'DROUGHT CONDITIONS', 'DROUGHT-RELATED', 'SEVERE DROUGHT',
    'EXTREME DROUGHT', 'PROLONGED DROUGHT', 'AGRICULTURAL DROUGHT', 'WATER SCARCITY',
    'WATER SHORTAGE', 'WATER CRISIS', 'DRY SPELL', 'DRYNESS', 'ARIDITY', 'DESICCATION',
    'FAMINE', 'FAMINE CONDITIONS', 'FAMINE-RELATED', 'SOIL MOISTURE DEFICIT',
    'GROUNDWATER DEPLETION', 'WATER STRESS', 'CLIMATE STRESS', 'IRRIGATION'
]

textual_keywords = [kw.lower() for kw in (flood_keywords + drought_keywords)]

# ========== Helpers ==========

def extract_district_state(location_str):
    if pd.isna(location_str):
        return None, None
    for loc in str(location_str).split(';'):
        parts = loc.split('#')
        if len(parts) >= 4:
            country_code = parts[2].upper()
            if country_code == 'IN':
                return parts[1], parts[3]
    return None, None

def parse_tone(tone_str):
    """
    Expect six comma-separated values in tone_str.
    Return tuple of floats or Nones if invalid.
    """
    try:
        parts = str(tone_str).split(',')
        if len(parts) == 6:
            return tuple(float(p) for p in parts)
    except Exception:
        pass
    return (None, None, None, None, None, None)

def download_gkg(date_str):
    url = f"http://data.gdeltproject.org/gkg/{date_str}.gkg.csv.zip"
    print("Downloading:", date_str, "->", url)
    r = requests.get(url)
    if r.status_code != 200:
        print("  Failed, status:", r.status_code)
        return None
    z = zipfile.ZipFile(io.BytesIO(r.content))
    for fname in z.namelist():
        if fname.lower().endswith('.csv'):
            z.extract(fname, gkg_folder)
            return os.path.join(gkg_folder, fname)
    print("  No CSV file found in zip")
    return None

def extract_matched_themes(themes_str, flood_kw, drought_kw):
    if not isinstance(themes_str, str):
        return ""
    parts = themes_str.split(';')
    matched = []
    for p in parts:
        p_up = p.upper()
        for kw in flood_kw:
            if kw.upper().replace(' ', '_') in p_up:
                matched.append(p)
                break
        else:
            for kw in drought_kw:
                if kw.upper().replace(' ', '_') in p_up:
                    matched.append(p)
                    break
    return ';'.join(matched)

def find_textual_mentions(quotes_str, docid, keywords):
    hits = []
    if isinstance(quotes_str, str):
        lower = quotes_str.lower()
        for kw in keywords:
            if kw in lower:
                hits.append(kw)
    if isinstance(docid, str):
        low = docid.lower()
        for kw in keywords:
            if kw in low:
                hits.append(kw)
    hits = list(sorted(set(hits)))
    return ';'.join(hits)

# ========== Main Loop ==========

all_records = []

current_date = start_date
while current_date <= end_date:
    date_str = current_date.strftime("%Y%m%d")
    csv_path = download_gkg(date_str)
    if not csv_path:
        current_date += timedelta(days=1)
        continue

    try:
        df = pd.read_csv(csv_path, sep='\t', dtype=str, low_memory=False)

        flood_pattern = '|'.join(flood_keywords)
        drought_pattern = '|'.join(drought_keywords)
        df['IsFlood'] = df['THEMES'].str.contains(flood_pattern, case=False, na=False)
        df['IsDrought'] = df['THEMES'].str.contains(drought_pattern, case=False, na=False)

        df_filtered = df[df['IsFlood'] | df['IsDrought']].copy()

        df_filtered[['District', 'State']] = df_filtered['LOCATIONS'].apply(
            lambda x: pd.Series(extract_district_state(x))
        )
        df_filtered = df_filtered.dropna(subset=['District', 'State'])

        def assign_event_type(r):
            if r['IsFlood']:
                return 'Flood'
            elif r['IsDrought']:
                return 'Drought'
            else:
                return None
        df_filtered['EventType'] = df_filtered.apply(assign_event_type, axis=1)

        df_filtered[['Tone_Avg', 'Tone_Positive', 'Tone_Negative', 'Tone_Polarity', 'Tone_ActivityRef', 'Tone_SelfGroupRef']] = \
            df_filtered['TONE'].apply(lambda x: pd.Series(parse_tone(x)))

        df_filtered['MatchedTheme'] = df_filtered['THEMES'].apply(
            lambda x: extract_matched_themes(x, flood_keywords, drought_keywords)
        )

        quote_col = None
        for col in df_filtered.columns:
            if col.upper().startswith('QUOTATION'):
                quote_col = col
                break

        docid_col = 'DOCUMENTIDENTIFIER' if 'DOCUMENTIDENTIFIER' in df_filtered.columns else None

        df_filtered['TextualMentions'] = df_filtered.apply(
            lambda r: find_textual_mentions(
                r.get(quote_col, ""), 
                r.get(docid_col, ""), 
                textual_keywords
            ), axis=1
        )

        # Include source fields (present in v1): SOURCES, SOURCEURLS
        output_cols = [
            'DATE', 'District', 'State', 'EventType', 'MatchedTheme',
            'SOURCES', 'SOURCEURLS',
            quote_col,
            'TextualMentions',
            'Tone_Avg', 'Tone_Positive', 'Tone_Negative', 'Tone_Polarity', 'Tone_ActivityRef', 'Tone_SelfGroupRef'
        ]

        output_cols = [c for c in output_cols if c in df_filtered.columns]

        sub = df_filtered[output_cols].copy()
        all_records.append(sub)

        print(date_str, "→", len(sub), "records matched and processed")

    except Exception as e:
        print("Error on", date_str, ":", e)

    try:
        os.remove(csv_path)
    except:
        pass

    current_date += timedelta(days=1)

if all_records:
    result_df = pd.concat(all_records, ignore_index=True)
    out_name = f"india_flood_drought_with_textual_jan_nov_2024.csv"
    out_path = os.path.join(gkg_folder, out_name)
    result_df.to_csv(out_path, index=False)
    print("✅ Saved result to:", out_path)
else:
    print("❌ No matching records found in the date range")


CLEANING

In [ ]:
import pandas as pd
import aiohttp
import asyncio
from urllib.parse import urlparse
import ssl
from aiohttp import ClientTimeout
from tqdm import tqdm
import re
import trafilatura

# ========== CONFIG ==========
CSV_PATH = r"C:\Users\dsp7\Box\MY RESEARCH\India Sentiment Project\GKG_Data\india_flood_drought_with_textual_jan_nov_2024.csv"
OUTPUT_PATH = CSV_PATH.replace(".csv", "_verified_filtered.csv")
BATCH_SIZE = 1000
SEMAPHORE_LIMIT = 10
TIMEOUT = 15

# ========== TRUSTED DOMAINS ==========
TRUSTED_DOMAINS = {
    "timesofindia.indiatimes.com", "thehindu.com", "indianexpress.com", "hindustantimes.com",
    "ndtv.com", "aninews.in", "business-standard.com", "livemint.com", "scroll.in",
    "news18.com", "tribuneindia.com", "theprint.in", "deccanherald.com",
    "newindianexpress.com", "dnaindia.com", "zeenews.india.com", "telegraphindia.com",
    "siasat.com", "outlookindia.com", "gujaratsamachar.com", "bhaskarenglish.in"
}

# ========== KEYWORDS ==========
flood_keywords = [
    'FLOOD', 'FLOODS', 'FLOODING', 'FLASH FLOOD', 'FLASH FLOODS', 'FLOODING EVENT',
    'FLOOD-RELATED', 'FLOOD DAMAGE', 'RIVER FLOOD', 'RIVER FLOODS', 'URBAN FLOODING',
    'MONSOON FLOOD', 'MONSOON FLOODS', 'HEAVY RAINFALL', 'HEAVY RAIN', 'INUNDATION',
    'WATER LOGGING', 'WATERLOGGING', 'FLOODING-RELATED'
]
drought_keywords = [
    'DROUGHT', 'DROUGHTS', 'DROUGHT CONDITIONS', 'DROUGHT-RELATED', 'SEVERE DROUGHT',
    'EXTREME DROUGHT', 'PROLONGED DROUGHT', 'AGRICULTURAL DROUGHT', 'WATER SCARCITY',
    'WATER SHORTAGE', 'WATER CRISIS', 'DRY SPELL', 'DRYNESS', 'ARIDITY', 'DESICCATION',
    'FAMINE', 'FAMINE CONDITIONS', 'FAMINE-RELATED', 'SOIL MOISTURE DEFICIT',
    'GROUNDWATER DEPLETION', 'WATER STRESS', 'CLIMATE STRESS', 'IRRIGATION'
]
keywords = [kw.lower() for kw in flood_keywords + drought_keywords]

# ========== UTILS ==========

def get_domain(url):
    try:
        parsed = urlparse(url)
        domain = parsed.netloc.lower()
        if domain.startswith("www."): domain = domain[4:]
        if domain.startswith("m."): domain = domain[2:]
        return domain
    except Exception:
        return ""

import re
import unicodedata
import html as html_lib

def clean_snippet(text, max_len=1500):
    # Decode HTML entities (e.g., &amp; → &)
    text = html_lib.unescape(text)

    # Remove HTML tags (e.g., <p>, </div>)
    text = re.sub(r'<[^>]+>', '', text)

    # Normalize unicode (e.g., fancy quotes → plain ones)
    text = unicodedata.normalize('NFKC', text)

    # Remove control characters (non-printable except \n and \t)
    text = ''.join(ch for ch in text if ch.isprintable() or ch in '\n\t')

    # Replace multiple whitespace characters with a single space
    text = re.sub(r'\s+', ' ', text)

    # Replace curly quotes and dashes with standard ones
    replacements = {
        '“': '"', '”': '"', '‘': "'", '’': "'",
        '–': '-', '—': '-', '…': '...',
    }
    for orig, repl in replacements.items():
        text = text.replace(orig, repl)

    # Strip leading/trailing whitespace and unnecessary punctuation
    text = text.strip(" \t\n\r\"'.,-–—")

    # Truncate without cutting mid-word if possible
    if len(text) > max_len:
        cutoff = text.rfind(' ', 0, max_len)
        text = text[:cutoff] if cutoff != -1 else text[:max_len]
        text += '...'

    return text

def keyword_match_and_extract(text, keyword_list):
    text= clean_snippet(text)
    for kw in keyword_list:
        if re.search(r'\b' + re.escape(kw) + r'\b', text):
            return kw, text
    return None, ""

# ========== ASYNC FETCH ==========

ssl_context = ssl.create_default_context()
timeout = ClientTimeout(total=TIMEOUT)
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/117.0 Safari/537.36"
}

async def fetch_and_check(url, session, semaphore):
    async with semaphore:
        try:
            async with session.get(url, ssl=ssl_context, timeout=timeout) as response:
                if response.status != 200:
                    return (url, False, "", "")
                html = await response.text()
                main_text = trafilatura.extract(html)
                if not main_text:
                    return (url, False, "", "")
                text = main_text.lower()
                match, snippet = keyword_match_and_extract(text, keywords)
                return (url, bool(match), snippet, match or "")
        except Exception:
            return (url, False, "", "")

async def process_batch(urls, session, semaphore, pbar):
    tasks = [fetch_and_check(url, session, semaphore) for url in urls]
    results = []
    for coro in tqdm(asyncio.as_completed(tasks), total=len(tasks), desc="🌐 Verifying", ncols=100):
        result = await coro
        results.append(result)
        
    return results

async def run_pipeline():
    df = pd.read_csv(CSV_PATH)
    df = df.dropna(subset=["SOURCEURLS"]).copy()

    df["domain"] = df["SOURCEURLS"].apply(get_domain)
    df = df[df["domain"].isin(TRUSTED_DOMAINS)]
    urls = df["SOURCEURLS"].tolist()

    print(f"🚀 Starting verification for {len(urls):,} trusted URLs...")

    semaphore = asyncio.Semaphore(SEMAPHORE_LIMIT)
    all_results = []
    pbar = tqdm(total=len(urls), desc="🌐 Verifying", ncols=100)

    async with aiohttp.ClientSession(headers=HEADERS) as session:
        for i in range(0, len(urls), BATCH_SIZE):
            batch = urls[i:i + BATCH_SIZE]
            results = await process_batch(batch, session, semaphore, pbar)
            all_results.extend(results)

    pbar.close()

    # Map results back to DataFrame
    results_dict = {url: (found, snippet, kw) for url, found, snippet, kw in all_results}
    df["has_keyword"] = df["SOURCEURLS"].map(lambda x: results_dict.get(x, (False, "", ""))[0])
    df["snippet"] = df["SOURCEURLS"].map(lambda x: results_dict.get(x, (False, "", ""))[1])
    df["matched_keyword"] = df["SOURCEURLS"].map(lambda x: results_dict.get(x, (False, "", ""))[2])

    # Filter only relevant
    df_filtered = df[df["has_keyword"]]

    print(f"✅ Found {len(df_filtered):,} verified relevant articles out of {len(df):,}")
    df_filtered.to_csv(OUTPUT_PATH, index=False)
    print("💾 Saved to:", OUTPUT_PATH)


FURTHER PROCESSING

In [ ]:
import pandas as pd
import requests
from newspaper import Article
from tqdm import tqdm

INPUT_CSV = r"C:\Users\dsp7\Box\MY RESEARCH\India Sentiment Project\GKG_Data\india_flood_drought_with_textual_jan_nov_2024_verified_filtered.csv"
OUTPUT_CSV = r"C:\Users\dsp7\Box\MY RESEARCH\India Sentiment Project\GKG_Data\india_with_article_text.csv"

df = pd.read_csv(INPUT_CSV)

def download_article_text(url):
    try:
        article = Article(url)
        article.download()
        article.parse()
        text = article.text
        if len(text) < 300:
            return None
        return text
    except Exception:
        return None

tqdm.pandas()
df["raw_text"] = df["SOURCEURLS"].progress_apply(download_article_text)

df = df.dropna(subset=["raw_text"])
df.to_csv(OUTPUT_CSV, index=False)

print(f"Saved scraped articles: {len(df)}")


In [ ]:
from nrclex import NRCLex

def nrc_scores(text):
    emotion = NRCLex(text)
    return emotion.raw_emotion_scores

df["nrc_emotions"] = df["raw_text"].apply(nrc_scores)

In [ ]:
emotions_df = df["nrc_emotions"].apply(pd.Series).fillna(0)
df = pd.concat([df, emotions_df], axis=1)


In [ ]:
import pandas as pd

# --- NRC emotion columns (emotions only) ---
emotion_cols = [
    "anger", "fear", "sadness", "joy",
    "disgust", "surprise", "anticipation", "trust"
]

# --- Total emotion words per article ---
df["emotion_total"] = df[emotion_cols].sum(axis=1)

# --- Emotion shares 
for e in emotion_cols:
    df[f"{e}_share"] = df[e] / df["emotion_total"]

# --- Handle articles with zero emotion words ---
share_cols = [f"{e}_share" for e in emotion_cols]
df[share_cols] = df[share_cols].fillna(0)


# --- Emotion intensity (normalized by article length) ---
df["word_count"] = df["raw_text"].str.split().str.len()

for e in emotion_cols:
    df[f"{e}_intensity"] = df[e] / df["word_count"]


# Emotion shares should sum to ~1 (or 0 if no emotion words)
df["emotion_share_sum"] = df[share_cols].sum(axis=1)

In [ ]:
NLP_CSV = r"C:\Users\dsp7\Box\MY RESEARCH\India Sentiment Project\GKG_Data\india_with_article_text_NLP.csv"
df.to_csv(NLP_CSV, index=False)